In [7]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd
from rasterio.mask import mask

# =========================
# Config
# =========================
TIF_2000 = r"../Result/Land_Cover_Each_Year/ESRI_2018_reclass_300m.tif"
TIF_2015 = r"../Result/Land_Cover_Each_Year/ESRI_2024_reclass_300m.tif"

OUT_TIF  = r"../Result/Land_Cover_Loss/loss_2018_2024_area_km2.tif"
OUT_CSV  = r"../Result/Land_Cover_Loss/loss_2018_2024_area_km2_stats.csv"

AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"  # for reference only

# --- nodata 用 255，避免和 0(No change) 冲突
NODATA_VAL = 255

LOSS_LABELS = {
    255: "NoData",
    0: "No change",
    1: "Tree-covered loss",
    2: "Grassland loss",
    3: "Cropland loss",
    4: "Wetland loss",
    5: "Artificial loss",
    6: "Other loss",
    7: "Water loss",
}

LOSS_COLORMAP = {
    255: (0, 0, 0, 0),          # NoData transparent
    0:   (255, 255, 230, 255),  # No change
    1:   (128, 128,   0, 255),  # Tree-covered loss
    2:   (255, 165,   0, 255),  # Grassland loss
    3:   (255, 255,   0, 255),  # Cropland loss
    4:   (  0, 255, 128, 255),  # Wetland loss (你可改色)
    5:   (255,   0,   0, 255),  # Artificial loss (你可改色)
    6:   (204, 153,  51, 255),  # Other loss
    7:   (  0, 102, 255, 255),  # Water loss
}


# =========================
# Helpers
# =========================
def _read_single_band(path):
    ds = rasterio.open(path)
    arr = ds.read(1)
    return ds, arr

def _ensure_same_grid(ds_a, arr_a, ds_b, arr_b):
    """
    If CRS/transform/shape differ, reproject B -> A using nearest neighbor.
    """
    same = (
        ds_a.crs == ds_b.crs and
        ds_a.transform == ds_b.transform and
        ds_a.width == ds_b.width and
        ds_a.height == ds_b.height
    )
    if same:
        return arr_a, arr_b, ds_a

    # Reproject B to match A
    dst = np.empty((ds_a.height, ds_a.width), dtype=arr_b.dtype)
    reproject(
        source=arr_b,
        destination=dst,
        src_transform=ds_b.transform,
        src_crs=ds_b.crs,
        dst_transform=ds_a.transform,
        dst_crs=ds_a.crs,
        resampling=Resampling.nearest,
        src_nodata=ds_b.nodata if ds_b.nodata is not None else NODATA_VAL,
        dst_nodata=NODATA_VAL,
    )
    return arr_a, dst, ds_a

def compute_loss(cls0, cls1, nodata=255):
    """
    Output codes:
      255 NoData
      0   No change
      1   Tree-covered loss (start=1)
      2   Grassland loss    (start=2)
      3   Cropland loss     (start=3)
      4   Wetland loss      (start=4)
      5   Artificial loss   (start=5)
      6   Other loss        (start=6)
      7   Water loss        (start=7)
    """
    cls0 = cls0.astype(np.int16, copy=False)
    cls1 = cls1.astype(np.int16, copy=False)

    valid = (cls0 != nodata) & (cls1 != nodata)

    # 默认先全是 NoData
    out = np.full(cls0.shape, fill_value=nodata, dtype=np.uint8)

    # No change
    same = valid & (cls0 == cls1)
    out[same] = 0

    # Loss: changed & start class defines loss type
    changed = valid & (cls0 != cls1)

    out[changed & (cls0 == 1)] = 1
    out[changed & (cls0 == 2)] = 2
    out[changed & (cls0 == 3)] = 3
    out[changed & (cls0 == 4)] = 4
    out[changed & (cls0 == 5)] = 5
    out[changed & (cls0 == 6)] = 6
    out[changed & (cls0 == 7)] = 7

    return out


def _pixel_area_km2_projected(ds):
    # Works when CRS unit is meters (most projected CRS)
    # Pixel area = |a * e| where a is pixel width, e is pixel height (negative)
    a = ds.transform.a
    e = ds.transform.e
    return abs(a * e) / 1e6

def _safe_nodata_for_dtype(nodata, dtype):
    """
    Ensure nodata is within dtype valid range.
    """
    dt = np.dtype(dtype)
    if np.issubdtype(dt, np.integer):
        info = np.iinfo(dt)
        n = int(nodata)
        if n < info.min or n > info.max:
            # fallback to 0 if possible; otherwise clamp
            if 0 >= info.min and 0 <= info.max:
                return 0
            return max(min(n, info.max), info.min)
        return n
    else:
        # float
        return float(nodata)

def _reproject_to_equal_area_for_area(ds, arr, ea_epsg=6933, nodata=255):
    """
    Reproject categorical raster to equal-area CRS for area stats.
    Force arr to int16 so nodata=0 is always valid.
    """
    # Force integer type to avoid nodata range issues
    src_arr = arr.astype(np.int16, copy=False)
    src_nodata = _safe_nodata_for_dtype(nodata, src_arr.dtype)

    dst_crs = rasterio.crs.CRS.from_epsg(ea_epsg)
    transform, width, height = calculate_default_transform(
        ds.crs, dst_crs, ds.width, ds.height, *ds.bounds
    )

    dst = np.full((height, width), src_nodata, dtype=np.int16)

    reproject(
        source=src_arr,
        destination=dst,
        src_transform=ds.transform,
        src_crs=ds.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
        src_nodata=src_nodata,
        dst_nodata=src_nodata,
    )

    pixel_area_km2 = abs(transform.a * transform.e) / 1e6
    return dst, transform, dst_crs, pixel_area_km2


def area_stats_km2(ds_ref, loss_arr, nodata=255):
    """
    Compute area by class code in km2.
    If ds_ref is projected (meters), use constant pixel area.
    Otherwise reproject to equal-area (EPSG:6933).
    """
    if ds_ref.crs is not None and ds_ref.crs.is_projected:
        px_km2 = abs(ds_ref.transform.a * ds_ref.transform.e) / 1e6
        vals, counts = np.unique(loss_arr, return_counts=True)
        areas = counts * px_km2
        return vals, areas
    else:
        ea_arr, _, _, px_km2 = _reproject_to_equal_area_for_area(
            ds_ref, loss_arr, ea_epsg=6933, nodata=nodata
        )
        vals, counts = np.unique(ea_arr, return_counts=True)
        areas = counts * px_km2
        return vals, areas

def load_aoi_geoms(aoi_shp, target_crs):
    gdf = gpd.read_file(aoi_shp)
    if gdf.empty:
        raise ValueError(f"AOI is empty: {aoi_shp}")
    if gdf.crs is None:
        raise ValueError("AOI has no CRS. Please define CRS for the AOI shapefile.")
    if target_crs is not None and gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    return [geom.__geo_interface__ for geom in gdf.geometry if geom is not None and not geom.is_empty]


# =========================
# Main
# =========================
os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

with rasterio.open(TIF_2000) as ds0, rasterio.open(TIF_2015) as ds1:

    # 1) AOI -> same CRS as rasters
    aoi_geoms = load_aoi_geoms(AOI_SHP, ds0.crs)

    # 2) mask/crop both rasters to AOI (AOI 外填 0)
    arr2000, out_transform = mask(ds0, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)
    arr2015, _             = mask(ds1, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)

    # arr is (bands, h, w)
    cls2000 = arr2000[0]
    cls2015 = arr2015[0]

    # 3) compute loss on AOI-cropped arrays
    loss = compute_loss(cls2000, cls2015, nodata=NODATA_VAL)

    # 4) write AOI-cropped loss tif
    profile = ds0.profile.copy()
    profile.update(
        height=loss.shape[0],
        width=loss.shape[1],
        transform=out_transform,
        count=1,
        dtype=rasterio.uint8,
        nodata=NODATA_VAL,
        compress="lzw",
        tiled=True,
    )

    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(loss.astype(np.uint8), 1)
        dst.write_colormap(1, LOSS_COLORMAP)

    print("Saved:", OUT_TIF)

    # 5) area stats (use a lightweight in-memory reference dataset-like object)
    # Trick: create a minimal "ds_ref" proxy using rasterio.open(OUT_TIF) for correct transform/bounds
with rasterio.open(OUT_TIF) as ds_ref:
    loss_arr = ds_ref.read(1)
    vals, areas = area_stats_km2(ds_ref, loss_arr, nodata=NODATA_VAL)

mask_valid = vals != NODATA_VAL
vals = vals[mask_valid]
areas = areas[mask_valid]

df = pd.DataFrame({
    "loss_code": vals.astype(int),
    "label": [LOSS_LABELS.get(int(v), "Unknown") for v in vals],
    "area_km2": areas
}).sort_values("loss_code")

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_CSV)
print(df)


Saved: ../Result/Land_Cover_Loss/loss_2018_2024_area_km2.tif
Saved: ../Result/Land_Cover_Loss/loss_2018_2024_area_km2_stats.csv
   loss_code              label      area_km2
0          0          No change  65048.095744
1          1  Tree-covered loss      2.647362
2          2     Grassland loss    788.368802
3          3      Cropland loss    161.878394
4          4       Wetland loss     16.818534
5          5    Artificial loss     61.979414
6          6         Other loss   5317.537838
7          7         Water loss     20.088805
